In [0]:
# Retail Sales Lakehouse - Audit & Error Logging

from pyspark.sql.functions import *
from datetime import datetime
import uuid

pipeline_name = "Retail_Sales_Lakehouse"
run_id = str(uuid.uuid4())
start_time = datetime.now()

print("Pipeline Name :", pipeline_name)
print("Run ID        :", run_id)
print("Start Time    :", start_time)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)
from datetime import datetime

end_time = datetime.now()

rows_processed = spark.table(
    "workspace.default.silver_customers"
).count()

# Explicit schema
audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_processed", LongType(), False),
    StructField("error_message", StringType(), True)
])

audit_data = [
    (
        run_id,
        pipeline_name,
        start_time,
        end_time,
        "SUCCESS",
        rows_processed,
        None
    )
]

audit_df = spark.createDataFrame(
    audit_data,
    schema=audit_schema
)

display(audit_df)

In [0]:
audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.pipeline_audit_log")

print("Audit log saved successfully")

In [0]:
audit_logs = spark.table(
    "workspace.default.pipeline_audit_log"
)

display(
    audit_logs.orderBy(
        col("start_time").desc()
    )
)

In [0]:
from datetime import datetime
import uuid

failure_run_id = str(uuid.uuid4())
failure_start_time = datetime.now()

try:
    # Intentional failure for testing
    spark.table("workspace.default.table_does_not_exist").count()

except Exception as e:
    failure_end_time = datetime.now()

    failure_audit_data = [
        (
            failure_run_id,
            pipeline_name,
            failure_start_time,
            failure_end_time,
            "FAILED",
            0,
            str(e)
        )
    ]

    failure_audit_df = spark.createDataFrame(
        failure_audit_data,
        schema=audit_schema
    )

    failure_audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("workspace.default.pipeline_audit_log")

    print("Failure captured and logged successfully")

In [0]:
from pyspark.sql.functions import col

display(
    spark.table("workspace.default.pipeline_audit_log")
    .select(
        "run_id",
        "pipeline_name",
        "start_time",
        "end_time",
        "status",
        "rows_processed",
        "error_message"
    )
    .orderBy(col("start_time").desc())
)